In [ ]:
pip install torch torchvision pandas Pillow tqdm scikit-learn

In [9]:
import os
DATA_DIR = '/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition'
# ลองดูโครงสร้างข้างใน train
print("Files in train dir:", os.listdir(os.path.join(DATA_DIR, 'train'))[:5])
# ลองดูว่ามีโฟลเดอร์ย่อยไหม
sub_dir = os.path.join(DATA_DIR, 'train', 'train')
if os.path.exists(sub_dir):
    print("Files in train/train dir:", os.listdir(sub_dir)[:5])

Files in train dir: ['train']
Files in train/train dir: ['img_13-8060598_100-6003712_a175-888775351385_s85-888775351385_y0_f90_1.jpg', 'img_13-7976009_100-5678585_a350-4302707_s80-4302707_y0_f90_1.jpg', 'img_13-7401085564025_100-529803635492_s270_f90_y0_1_09-2020.jpg', 'img_13-8379284_100-5919369_a269-8149035_s179-8149035_y0_f90_1.jpg', 'MrtSutthisan_img_13-7864989_100-5658525_a177_s90_y75_f90_0.jpg']


In [12]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# --- 1. Configuration ---
DATA_DIR = '/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition'
if not os.path.exists(DATA_DIR):
    DATA_DIR = '.'

TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train', 'train')
TEST_IMG_DIR = os.path.join(DATA_DIR, 'test', 'test')
if not os.path.exists(TEST_IMG_DIR):
    TEST_IMG_DIR = os.path.join(DATA_DIR, 'test')

TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
SAMPLE_SUB = os.path.join(DATA_DIR, 'sample_submission.csv')

IMG_SIZE = 224
BATCH_SIZE = 16 # ลด Batch Size เล็กน้อยเพื่อความเสถียร
EPOCHS = 20     # เพิ่ม Epoch
LEARNING_RATE = 5e-5 # LR ต่ำลงสำหรับ Fine-tune ลึก
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {DEVICE}")

# --- 2. Smart File Mapping (เหมือนเดิมแต่ชัวร์) ---
def build_smart_map(directory):
    file_map = {}
    if not os.path.exists(directory):
        return file_map
    for filename in os.listdir(directory):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            full_path = os.path.join(directory, filename)
            file_map[filename] = full_path
            if 'img_' in filename:
                suffix = filename[filename.find('img_'):]
                file_map[suffix] = full_path
            name_no_ext = os.path.splitext(filename)[0]
            file_map[name_no_ext] = full_path
            if 'img_' in name_no_ext:
                suffix_no_ext = name_no_ext[name_no_ext.find('img_'):]
                file_map[suffix_no_ext] = full_path
    return file_map

train_map = build_smart_map(TRAIN_IMG_DIR)
test_map = build_smart_map(TEST_IMG_DIR)

# --- 3. Dataset ---
class HouseDataset(Dataset):
    def __init__(self, df, file_map, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.file_map = file_map
        self.transform = transform
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        key = str(row['id']) if self.is_test else str(row['image_name'])
        
        img_path = self.file_map.get(key, None)
        if img_path is None and 'img_' in key:
            suffix = key[key.find('img_'):]
            img_path = self.file_map.get(suffix, None)
            
        if img_path and os.path.exists(img_path):
            try:
                img = Image.open(img_path).convert('RGB')
            except:
                img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=(128, 128, 128))
        else:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=(128, 128, 128))
            
        if self.transform:
            img = self.transform(img)
            
        if self.is_test:
            return img, str(row['id'])
        else:
            return img, torch.tensor(float(row['class']), dtype=torch.float32)

# --- 4. Transforms for TTA ---
# Transform ปกติสำหรับ Train
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Transform สำหรับ Validation/Test (พื้นฐาน)
base_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Transform สำหรับ TTA (กลับด้าน)
flip_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=1.0), # บังคับกลับด้าน
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 5. Load Data ---
train_df = pd.read_csv(TRAIN_CSV)
sub_df = pd.read_csv(SAMPLE_SUB)

tr_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['class'])

train_ds = HouseDataset(tr_df, train_map, train_tf)
val_ds = HouseDataset(val_df, train_map, base_tf) # ใช้ base_tf สำหรับ Val เพื่อหา Threshold
test_ds = HouseDataset(sub_df, test_map, base_tf, is_test=True) # ใช้ base_tf เป็นหลัก

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
# สร้าง Loader สำหรับ TTA แยก
test_ds_flip = HouseDataset(sub_df, test_map, flip_tf, is_test=True)
test_loader_flip = DataLoader(test_ds_flip, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# --- 6. Model (ResNet50 - Deeper Fine-tuning) ---
model = models.resnet50(pretrained=True)
# Unfreeze Layer 3 และ 4
for param in model.parameters():
    param.requires_grad = False
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True

num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_ftrs, 1),
    nn.Sigmoid()
)
model = model.to(DEVICE)

criterion = nn.BCELoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# --- 7. Training ---
best_acc = 0.0
for epoch in range(EPOCHS):
    model.train()
    t_loss, t_corr, t_tot = 0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f'Train {epoch+1}'):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, labels)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * imgs.size(0)
        preds = (outs > 0.5).float()
        t_corr += (preds == labels).sum().item()
        t_tot += labels.size(0)
    
    model.eval()
    v_corr, v_tot = 0, 0
    val_probs = [] # เก็บ Probability ไว้หา Threshold
    val_labels = []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f'Val {epoch+1}'):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
            outs = model(imgs)
            preds = (outs > 0.5).float()
            v_corr += (preds == labels).sum().item()
            v_tot += labels.size(0)
            val_probs.extend(outs.cpu().numpy().flatten())
            val_labels.extend(labels.cpu().numpy().flatten())
            
    v_acc = v_corr / v_tot
    print(f"Ep {epoch+1}/{EPOCHS} | Train Acc: {t_corr/t_tot:.4f} | Val Acc: {v_acc:.4f}")
    
    if v_acc > best_acc:
        best_acc = v_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"=> Saved Best: {best_acc:.4f}")

# --- 8. Find Best Threshold ---
print("Finding optimal threshold...")
best_thresh = 0.5
best_val_acc_thresh = 0.0
val_probs = np.array(val_probs)
val_labels = np.array(val_labels)

for thresh in np.arange(0.3, 0.7, 0.01):
    preds = (val_probs >= thresh).astype(int)
    acc = (preds == val_labels).mean()
    if acc > best_val_acc_thresh:
        best_val_acc_thresh = acc
        best_thresh = thresh

print(f"Best Threshold: {best_thresh:.2f} with Acc: {best_val_acc_thresh:.4f}")

# --- 9. Prediction with TTA (Fixed) ---
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

def get_predictions(loader, device):
    """ฟังก์ชันช่วยทำนายผล คืนค่า Dictionary {id: probability}"""
    predictions = {}
    with torch.no_grad():
        for imgs, ids in tqdm(loader, desc="Predicting", leave=False):
            imgs = imgs.to(device)
            outs = model(imgs)
            probs = outs.cpu().numpy().flatten()
            
            # จับคู่ ID กับ Probability
            for img_id, prob in zip(ids, probs):
                predictions[img_id] = prob
    return predictions

print("Predicting Base Images...")
preds_base = get_predictions(test_loader, DEVICE)

print("Predicting Flipped Images...")
preds_flip = get_predictions(test_loader_flip, DEVICE)

# รวมผลลัพธ์โดยเฉลี่ยความน่าจะเป็น
all_ids = []
all_avg_probs = []

# วนลูปตามลำดับใน sample_submission เพื่อความชัวร์
for img_id in sub_df['id']:
    id_str = str(img_id)
    # ดึงค่าจาก dict ถ้าไม่มีให้ใช้ 0.5 (กรณี error หายากมาก)
    p1 = preds_base.get(id_str, 0.5)
    p2 = preds_flip.get(id_str, 0.5)
    
    avg_prob = (p1 + p2) / 2.0
    all_ids.append(id_str)
    all_avg_probs.append(avg_prob)

# --- Find Best Threshold (ใช้ข้อมูลจาก Validation เดิม) ---
# (สมมติว่าคุณมีตัวแปร val_probs และ val_labels จากขั้นตอน Training ก่อนหน้า)
# ถ้าไม่มี ให้ใช้ 0.5 ไปก่อน หรือรันส่วนหา Threshold แยก
best_thresh = 0.5 
# หากคุณมีโค้ดส่วนหา Threshold จากครั้งก่อน ให้ใส่ไว้ที่นี่
# else:
#     print("Using default threshold 0.5")

# Convert Prob to Answer
final_preds = (np.array(all_avg_probs) >= best_thresh).astype(int)

submission = pd.DataFrame({'id': all_ids, 'answer': final_preds})

# ตรวจสอบความถูกต้องของลำดับอีกครั้ง
original_order = sub_df['id'].astype(str)
submission['id'] = submission['id'].astype(str)
submission = submission.set_index('id').loc[original_order].reset_index()

submission.to_csv('submission.csv', index=False)
print("Submission saved!")
print(submission.head())
print(f"Prediction Distribution:\n{submission['answer'].value_counts()}")

Device: cuda


Val 1: 100%|██████████| 19/19 [00:03<00:00,  5.44it/s]


Ep 1/20 | Train Acc: 0.8957 | Val Acc: 0.9696
=> Saved Best: 0.9696


Val 2: 100%|██████████| 19/19 [00:03<00:00,  5.53it/s]


Ep 2/20 | Train Acc: 0.9458 | Val Acc: 0.9358


...

Ep 19/20 | Train Acc: 0.9944 | Val Acc: 0.9628


Val 20: 100%|██████████| 19/19 [00:03<00:00,  5.77it/s]


Ep 20/20 | Train Acc: 0.9951 | Val Acc: 0.9628
Finding optimal threshold...
Best Threshold: 0.49 with Acc: 0.9628
Predicting Base Images...


Predicting Flipped Images...


Submission saved!
         id  answer
0  e4b420b0       0
1  23efa479       0
2  1f0f2402       0
3  8a60480c       0
4  11f20127       0
Prediction Distribution:
answer
0    811
1    739
Name: count, dtype: int64
